<a href="https://colab.research.google.com/github/vee-16/contextual-deepfake-detection/blob/benchmarks/prithivMLmods_evals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers datasets pillow scikit-learn tqdm

import torch
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoImageProcessor, AutoModelForImageClassification
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from PIL import Image
import os

In [5]:
model_name = "prithivMLmods/Deep-Fake-Detector-v2-Model"

processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModelForImageClassification.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

print("Model loaded on:", device)

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Model loaded on: cuda


In [6]:
from datasets import load_dataset, Image as HFImage

dataset = load_dataset(
    "ComplexDataLab/OpenFake",
    split="test",
    streaming=True
)

# IMPORTANT: prevents Hugging Face from auto-opening images before our try/except
dataset = dataset.cast_column("image", HFImage(decode=False))

print(dataset)

Resolving data files:   0%|          | 0/206 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/206 [00:00<?, ?it/s]

IterableDataset({
    features: ['image', 'prompt', 'label', 'model', 'type', 'release_date'],
    num_shards: 7
})


In [7]:
label_map = {
    "real": 0,
    "fake": 1,
    "Real": 0,
    "Fake": 1,
    "Realism": 0,
    "Deepfake": 1
}

In [9]:
from PIL import Image, UnidentifiedImageError
import io
import torch
import numpy as np
from tqdm import tqdm

batch_size = 32  # try 16, 32, or 64 depending on GPU memory

y_true = []
y_pred = []
y_prob = []

skipped = 0
batch_images = []
batch_labels = []

def flush_batch(batch_images, batch_labels):
    if len(batch_images) == 0:
        return

    inputs = processor(
        images=batch_images,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()

    pred_idxs = np.argmax(probs, axis=1)

    for i, pred_idx in enumerate(pred_idxs):
        pred_label = model.config.id2label[int(pred_idx)]
        pred = label_map.get(pred_label, 1)

        y_true.append(batch_labels[i])
        y_pred.append(pred)
        y_prob.append(probs[i][1])  # fake probability

for sample in tqdm(dataset):
    try:
        raw_image = sample["image"]

        if isinstance(raw_image, dict) and raw_image.get("bytes") is not None:
            image = Image.open(io.BytesIO(raw_image["bytes"])).convert("RGB")
        elif isinstance(raw_image, dict) and raw_image.get("path") is not None:
            image = Image.open(raw_image["path"]).convert("RGB")
        else:
            image = raw_image.convert("RGB")

        true = label_map[sample["label"]]

        batch_images.append(image)
        batch_labels.append(true)

        if len(batch_images) == batch_size:
            flush_batch(batch_images, batch_labels)
            batch_images = []
            batch_labels = []

    except (UnidentifiedImageError, OSError, KeyError, TypeError):
        skipped += 1
        continue

# process final leftover batch
flush_batch(batch_images, batch_labels)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

print("Finished inference")
print("Total evaluated:", len(y_true))
print("Skipped images:", skipped)

8015it [03:19, 46.98it/s]/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
29459it [12:08, 47.78it/s]/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 2. 
  warnings.warn(str(msg))
59658it [24:39, 40.33it/s]

Finished inference
Total evaluated: 59655
Skipped images: 3


In [10]:
print("y_true shape:", y_true.shape)
print("y_pred shape:", y_pred.shape)
print("y_prob shape:", y_prob.shape)

print("Unique true labels:", np.unique(y_true, return_counts=True))
print("Unique predicted labels:", np.unique(y_pred, return_counts=True))

y_true shape: (59655,)
y_pred shape: (59655,)
y_prob shape: (59655,)
Unique true labels: (array([0, 1]), array([29826, 29829]))
Unique predicted labels: (array([0, 1]), array([ 1083, 58572]))


In [11]:
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

try:
    roc = roc_auc_score(y_true, y_prob)
except ValueError:
    roc = float("nan")

cm = confusion_matrix(y_true, y_pred)

print("\n=== HF Model Metrics on OpenFake Test Set ===")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {roc:.4f}")
print("\nConfusion Matrix rows=true, cols=pred:")
print(cm)


=== HF Model Metrics on OpenFake Test Set ===
Accuracy : 0.5028
Precision: 0.5014
Recall   : 0.9846
F1-score : 0.6644
ROC-AUC  : 0.4305

Confusion Matrix rows=true, cols=pred:
[[  623 29203]
 [  460 29369]]


In [14]:
import csv


csv_path = os.path.join( "prithivMLmods_evals.csv")

cm_flat = cm.flatten()

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "accuracy", "precision", "recall", "f1_score", "roc_auc",
        "cm_00", "cm_01", "cm_10", "cm_11"
    ])
    writer.writerow([
        f"{acc:.6f}",
        f"{prec:.6f}",
        f"{rec:.6f}",
        f"{f1:.6f}",
        f"{roc:.6f}",
        cm_flat[0], cm_flat[1], cm_flat[2], cm_flat[3]
    ])

print("Saved metrics to:", csv_path)

Saved metrics to: prithivMLmods_evals.csv
